# 01 · Data understanding and preprocessing

Goal of this notebook: understand VisDrone-VID before training anything, and produce the artefacts every later notebook relies on.

**Outputs** (written to `OUT`): `folds.csv` (sequence-level folds), `anchors.json` (k-means anchors), `eda_summary.json` (key numbers quoted in the report), and the Ultralytics dataset for the development fold (`yolo_fold0/data.yaml`).

Sections: 1 inventory · 2 annotation check · 3 classes · 4 object sizes · 5 density · 6 occlusion · 7 ignored regions · 8 temporal redundancy · 9 anchors · 10 folds · 11 preprocessing export · 12 evaluator sanity check · 13 conclusions.

## 0. Setup

In [ ]:
# On Kaggle: clone the project repo so that src/vdet.py is importable.
import os, sys
from pathlib import Path

REPO = Path("/kaggle/working/repo") if Path("/kaggle").exists() else Path("..").resolve()
if Path("/kaggle").exists() and not REPO.exists():
    !git clone https://github.com/<user>/<repo>.git {REPO}
sys.path.insert(0, str(REPO / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import vdet
from vdet import CLASSES

vdet.seed_everything(vdet.SEED)
ROOT = vdet.find_data_root()
OUT = Path("/kaggle/working") if Path("/kaggle").exists() else REPO / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160, "display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("data root:", ROOT, "| outputs:", OUT)

In [ ]:
# Load both labelled splits from COCO json (single source of truth for boxes).
imgs, anns = vdet.load_all(ROOT)
gt = anns[anns.iscrowd == 0].copy()
ign = anns[anns.iscrowd == 1].copy()
test = pd.read_csv(ROOT / "test_images.csv")
gt["cls_name"] = gt.cls.map(dict(enumerate(CLASSES)))
gt["sqrt_area"] = np.sqrt(gt.w * gt.h)
print(f"frames: {len(imgs):,}  boxes: {len(gt):,}  ignored regions: {len(ign):,}  test frames: {len(test):,}")
anns.head(3)

## 1. Inventory: clips, frames and resolutions

In [ ]:
clip_tab = (imgs.groupby(["split", "clip"])
            .agg(frames=("key", "size"), width=("width", "first"), height=("height", "first"))
            .reset_index())
test_tab = test.groupby("clip").agg(frames=("image_id", "size"), width=("width", "first"),
                                    height=("height", "first")).reset_index().assign(split="test")
all_clips = pd.concat([clip_tab, test_tab], ignore_index=True)
all_clips["res"] = all_clips.width.astype(str) + "x" + all_clips.height.astype(str)

display(all_clips.groupby("split").agg(clips=("clip", "size"), frames=("frames", "sum"),
                                       min_len=("frames", "min"), median_len=("frames", "median"),
                                       max_len=("frames", "max")))
res = all_clips.pivot_table(index="res", columns="split", values="frames", aggfunc="sum", fill_value=0)
res = res / res.sum()
display(res.sort_values("train", ascending=False).style.format("{:.1%}"))

Resolution matters because a fixed network input size (e.g. 640) shrinks 1920-wide frames three times more than 680-wide ones. Check whether the test resolution mix matches train: if it does not, per-resolution validation results (Section 4) are more trustworthy than the overall number.

## 2. Annotation format check (COCO vs YOLO txt, visual)

In [ ]:
# Numeric check: YOLO labels and ignore files must match the COCO json on random frames.
rng = np.random.default_rng(vdet.SEED)
sample = imgs.iloc[rng.choice(len(imgs), size=min(300, len(imgs)), replace=False)]
gt_by_key = dict(tuple(gt.groupby("key"))); ign_by_key = dict(tuple(ign.groupby("key")))
max_err, n_mismatch = 0.0, 0
for r in sample.itertuples():
    y = vdet.read_yolo_txt(vdet.label_path(r.img_path, "labels"), r.width, r.height)
    c = gt_by_key.get(r.key, gt.iloc[:0])
    yi = vdet.read_yolo_txt(vdet.label_path(r.img_path, "ignore"), r.width, r.height, has_cls=False)
    ci = ign_by_key.get(r.key, ign.iloc[:0])
    if len(y) != len(c) or len(yi) != len(ci):
        n_mismatch += 1; continue
    if len(y):
        a = np.sort(y, axis=0); b = np.sort(np.c_[c.cls, c[["x", "y", "w", "h"]]], axis=0)
        max_err = max(max_err, np.abs(a - b).max())
print(f"frames checked: {len(sample)}, count mismatches: {n_mismatch}, max coord diff: {max_err:.2f}px")

In [ ]:
COLORS = plt.cm.tab10(np.arange(len(CLASSES)))

def show_frame(row, ax, crop=None):
    img = cv2.cvtColor(cv2.imread(row.img_path), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    for b in gt_by_key.get(row.key, gt.iloc[:0]).itertuples():
        ax.add_patch(patches.Rectangle((b.x, b.y), b.w, b.h, fill=False, lw=0.8, ec=COLORS[b.cls]))
    for b in ign_by_key.get(row.key, ign.iloc[:0]).itertuples():
        ax.add_patch(patches.Rectangle((b.x, b.y), b.w, b.h, fill=True, alpha=0.25, fc="red", ec="red", hatch="//"))
    if crop is not None:
        ax.set_xlim(crop[0], crop[2]); ax.set_ylim(crop[3], crop[1])
    n = len(gt_by_key.get(row.key, []))
    ax.set_title(f"{row.clip_uid} #{row.frame}  ({row.width}x{row.height}, {n} obj)", fontsize=8)
    ax.axis("off")

# one frame from 6 different clips; prefer frames that have ignored regions
cand = imgs[imgs.key.isin(ign_by_key)].groupby("clip_uid").head(1)
cand = pd.concat([cand, imgs.groupby("clip_uid").head(1)]).drop_duplicates("clip_uid").head(6)
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for r, ax in zip(cand.itertuples(), axes.ravel()):
    show_frame(r, ax)
handles = [patches.Patch(color=COLORS[i], label=n) for i, n in enumerate(CLASSES)]
handles.append(patches.Patch(color="red", alpha=0.3, label="ignored"))
fig.legend(handles=handles, loc="lower center", ncol=9); plt.tight_layout(); plt.show()

In [ ]:
# Zoom into the densest frame to judge how small/packed objects really are
dense = gt.groupby("key").size().idxmax()
row = next(imgs[imgs.key == dense].itertuples())
g = gt_by_key[dense]
cx, cy = g.x.median(), g.y.median(); s = min(row.width, row.height) / 3
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
show_frame(row, axes[0]); show_frame(row, axes[1], crop=(cx - s, cy - s / 1.7, cx + s, cy + s / 1.7))
plt.tight_layout(); plt.show()

## 3. Class distribution

In [ ]:
cls_tab = gt.groupby(["cls_name", "split"]).size().unstack(fill_value=0).reindex(CLASSES)
cls_share = cls_tab / cls_tab.sum()
display(pd.concat([cls_tab, cls_share.add_suffix(" %")], axis=1)
        .style.format("{:.1%}", subset=[c + " %" for c in cls_tab.columns]))

# per-clip class mix: how much does it vary between sequences?
per_clip = gt.groupby(["clip_uid", "cls_name"]).size().unstack(fill_value=0)[CLASSES]
per_clip = per_clip.div(per_clip.sum(axis=1), axis=0)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 2.2]})
cls_tab.plot.barh(ax=axes[0], logx=True); axes[0].set_title("boxes per class (log)")
per_clip.sort_values("car").plot.bar(stacked=True, ax=axes[1], width=0.9, color=COLORS, legend=False)
axes[1].set_title("class share per clip"); axes[1].set_xticklabels([]); axes[1].set_xlabel("clip")
plt.tight_layout(); plt.show()
print("clips without a given class:\n", (per_clip == 0).sum().to_string())

## 4. Object sizes

In [ ]:
# COCO size buckets use area in original pixels: small < 32^2, medium < 96^2
gt["bucket"] = pd.cut(gt.w * gt.h, [0, 32**2, 96**2, np.inf], labels=["small", "medium", "large"])
size_tab = gt.groupby(["cls_name", "bucket"], observed=False).size().unstack().reindex(CLASSES)
size_tab = size_tab.div(size_tab.sum(axis=1), axis=0)
size_tab["median_sqrt_area_px"] = gt.groupby("cls_name").sqrt_area.median().reindex(CLASSES)
overall = gt.bucket.value_counts(normalize=True).reindex(["small", "medium", "large"])
print("overall:", overall.round(3).to_dict())
display(size_tab.style.format("{:.1%}", subset=["small", "medium", "large"]).format("{:.1f}", subset=["median_sqrt_area_px"]))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
bins = np.logspace(0, np.log10(gt.sqrt_area.max() + 1), 60)
for c, n in enumerate(CLASSES):
    axes[0].hist(gt.sqrt_area[gt.cls == c], bins=bins, histtype="step", color=COLORS[c], label=n, density=True)
axes[0].set_xscale("log"); axes[0].axvline(32, c="k", ls="--"); axes[0].set_title("sqrt(area), px"); axes[0].legend(fontsize=7)
axes[1].hexbin(gt.w, gt.h, gridsize=60, bins="log", xscale="log", yscale="log", mincnt=1)
axes[1].set_xlabel("w px"); axes[1].set_ylabel("h px"); axes[1].set_title("box w vs h")
gt.assign(res=gt.width.astype(str) + "x" + gt.height.astype(str)).boxplot("sqrt_area", by="res", ax=axes[2], showfliers=False, rot=30)
axes[2].set_title("object size by frame resolution"); plt.suptitle(""); plt.tight_layout(); plt.show()

In [ ]:
# What happens to objects after letterbox-resizing the long side to the network input size?
long_side = np.maximum(gt.width, gt.height)
rows = []
for s in [640, 960, 1280, 1536]:
    sc = np.minimum(s / long_side, 1.0)
    side = gt.sqrt_area * sc
    rows.append({"imgsz": s, "median_px": side.median(), "<8px": (side < 8).mean(),
                 "<16px": (side < 16).mean(), "stride8 cells/obj": (side / 8).median()})
display(pd.DataFrame(rows).style.format({"median_px": "{:.1f}", "<8px": "{:.1%}", "<16px": "{:.1%}", "stride8 cells/obj": "{:.2f}"}))

Objects below ~8 px after resizing cover less than one cell of a stride-8 feature map (P3), so they are close to undetectable for standard heads. This table is the main argument for the input resolution, for adding a P2 (stride-4) head and for slicing inference (SAHI) later.

## 5. Scene density

In [ ]:
cnt = gt.groupby("key").size().reindex(imgs.key, fill_value=0)
imgs["n_obj"] = cnt.values
print(imgs.n_obj.describe(percentiles=[.5, .9, .99]).round(1).to_string())
print(f"frames with >100 objects: {(imgs.n_obj > 100).mean():.1%}, >300: {(imgs.n_obj > 300).mean():.2%}, >500: {(imgs.n_obj > 500).mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(16, 4), gridspec_kw={"width_ratios": [1, 2.5]})
axes[0].hist(imgs.n_obj, bins=60); axes[0].set_title("objects per frame")
order = imgs.groupby("clip_uid").n_obj.median().sort_values().index
axes[1].boxplot([imgs.n_obj[imgs.clip_uid == c] for c in order], showfliers=False)
axes[1].set_xticks([]); axes[1].set_xlabel("clip (sorted by median)"); axes[1].set_title("objects per frame by clip")
plt.tight_layout(); plt.show()

# crowdedness: share of boxes overlapping another box with IoU > 0.5 (what NMS at 0.5 could wrongly suppress)
def overlap_share(g):
    b = g[["x", "y", "w", "h"]].values
    if len(b) < 2: return 0.0
    x1 = np.maximum(b[:, None, 0], b[None, :, 0]); y1 = np.maximum(b[:, None, 1], b[None, :, 1])
    x2 = np.minimum(b[:, None, 0] + b[:, None, 2], b[None, :, 0] + b[None, :, 2])
    y2 = np.minimum(b[:, None, 1] + b[:, None, 3], b[None, :, 1] + b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    a = b[:, 2] * b[:, 3]; iou = inter / (a[:, None] + a[None] - inter); np.fill_diagonal(iou, 0)
    return float((iou.max(1) > 0.5).mean())
sub_keys = imgs.key.sample(min(500, len(imgs)), random_state=vdet.SEED)
ov = np.mean([overlap_share(gt_by_key[k]) for k in sub_keys if k in gt_by_key])
print(f"boxes with a neighbour at IoU>0.5: {ov:.2%}")

## 6. Occlusion and truncation

In [ ]:
if {"occlusion", "truncation"} <= set(gt.columns):
    occ = gt.groupby(["cls_name", "occlusion"]).size().unstack(fill_value=0).reindex(CLASSES)
    display(occ.div(occ.sum(axis=1), axis=0).style.format("{:.1%}").set_caption("occlusion level per class (VisDrone: 0 none, 1 partial, 2 heavy)"))
    print("truncated share:", round((gt.truncation > 0).mean(), 3))

## 7. Ignored regions

In [ ]:
def covered_fraction(g, W, H, scale=4):
    m = np.zeros((int(H) // scale + 1, int(W) // scale + 1), bool)
    for b in g.itertuples():
        m[int(b.y) // scale:int(b.y + b.h) // scale + 1, int(b.x) // scale:int(b.x + b.w) // scale + 1] = True
    return m.mean()

ign_frames = imgs[imgs.key.isin(ign_by_key)]
cov = [covered_fraction(ign_by_key[r.key], r.width, r.height) for r in ign_frames.sample(min(1000, len(ign_frames)), random_state=0).itertuples()]
print(f"frames with ignored regions: {len(ign_frames) / len(imgs):.1%}")
print(f"area covered (frames that have them): median {np.median(cov):.1%}, p90 {np.quantile(cov, .9):.1%}")
print(f"regions per such frame: {ign.groupby('key').size().median():.0f} (median)")
per_clip_ign = imgs.assign(has=imgs.key.isin(ign_by_key)).groupby("clip_uid").has.mean()
print(f"clips where >50% of frames have ignored regions: {(per_clip_ign > .5).sum()} / {len(per_clip_ign)}")

Handling decision: ignored regions are **grey-filled in training images** (pixels of annotated boxes overlapping them are restored), so the detector is never taught that crowds/parking lots are background. In evaluation they are **crowd boxes for every class**, so detections inside them are neither TP nor FP, matching the competition rule. The non-target VisDrone classes (*ignored*, *others*) are already folded into these regions by the organisers.

## 8. Temporal redundancy (justifying frame subsampling)

In [ ]:
# Same track_id at frame t and t+k: how much does the box change?
tr = gt[gt.track_id.notna()][["clip_uid", "track_id", "frame_idx", "x", "y", "w", "h"]] if "track_id" in gt else None
rows = []
if tr is not None and len(tr):
    for k in [1, 2, 3, 5, 10, 20, 40]:
        m = tr.merge(tr.assign(frame_idx=tr.frame_idx - k), on=["clip_uid", "track_id", "frame_idx"], suffixes=("", "_k"))
        if len(m) == 0: continue
        iou = vdet.iou_xywh(m[["x", "y", "w", "h"]].values, m[["x_k", "y_k", "w_k", "h_k"]].values)
        shift = np.hypot(m.x_k - m.x, m.y_k - m.y) / np.sqrt(m.w * m.h)
        rows.append({"lag": k, "median IoU": np.median(iou), "IoU>0.7": (iou > .7).mean(), "median shift / size": np.median(shift)})
    tem = pd.DataFrame(rows); display(tem.style.format("{:.3f}", subset=tem.columns[1:]))
    plt.figure(figsize=(6, 3)); plt.plot(tem.lag, tem["median IoU"], "o-"); plt.xscale("log")
    plt.xlabel("frame lag"); plt.ylabel("median IoU of same track"); plt.show()

If the median IoU of the same object is still high at lag *k*, frames *k* apart are near-duplicates for the detector. Training on every *k*-th frame then costs little accuracy but saves *k*× GPU time per epoch; it also lets us spend the saved budget on higher input resolution. The chosen `STEP` lives in `vdet.PROTOCOL` (shared by all notebooks); edit it there after reading this table.

## 9. Anchor shapes from data (k-means, 1 − IoU)

In [ ]:
IMGSZ = vdet.PROTOCOL["imgsz"]  # input long side used for training (Section 4); anchors are expressed at this scale
sc = np.minimum(IMGSZ / np.maximum(gt.width, gt.height), 1.0).values
wh = np.c_[gt.w.values * sc, gt.h.values * sc]
wh_s = wh[np.random.default_rng(0).choice(len(wh), min(200_000, len(wh)), replace=False)]
anchors, mean_iou = vdet.kmeans_anchors(wh_s, k=9)
print(f"k=9 anchors at imgsz={IMGSZ} (w x h, px):"); print(np.round(anchors, 1))
print(f"mean best-anchor IoU: {mean_iou:.3f}")
ks = [3, 5, 9, 12, 15]
curve = [vdet.kmeans_anchors(wh_s, k=k)[1] for k in ks]
print("k -> mean IoU:", dict(zip(ks, np.round(curve, 3))))
print("aspect ratio h/w quantiles:", np.round(np.quantile(wh_s[:, 1] / wh_s[:, 0], [.1, .5, .9]), 2))
json.dump({"imgsz": IMGSZ, "anchors_wh": anchors.round(2).tolist(), "mean_iou": mean_iou}, open(OUT / "anchors.json", "w"), indent=1)

## 10. Validation design: sequence-level folds

Frames from one clip are near-duplicates (Section 8), so a frame-level split would leak. We pool the 63 labelled clips (train + val) and run `GroupKFold(5)` with the clip as group. **Fold 0 is the fixed development holdout** for all model comparisons (one fold per experiment fits the GPU budget); the remaining folds are used for out-of-fold checks of the final pipeline. The organisers' `val/` clips are spread across folds rather than used alone, because 7 clips is too few for stable small-class AP.

In [ ]:
folds = vdet.make_folds(imgs, n_splits=vdet.PROTOCOL["n_folds"])
folds.to_csv(OUT / "folds.csv", index=False)
fi = imgs.merge(folds, on="clip_uid"); fg = gt.merge(folds, on="clip_uid")
fold_tab = fi.groupby("fold").agg(clips=("clip_uid", "nunique"), frames=("key", "size"), obj_per_frame=("n_obj", "mean"))
fold_tab["small %"] = fg.groupby("fold").bucket.apply(lambda b: (b == "small").mean())
cls_fold = fg.groupby(["fold", "cls_name"]).size().unstack(fill_value=0)[CLASSES]
display(pd.concat([fold_tab, cls_fold.div(cls_fold.sum(axis=1), axis=0)], axis=1).style.format("{:.1%}", subset=["small %"] + CLASSES).format("{:.1f}", subset=["obj_per_frame"]))

If one fold is clearly off (e.g. no buses), reorder with a different `n_splits` or swap a clip; record any such change in the report.

## 11. Preprocessing export (Ultralytics layout, fold 0)

In [ ]:
STEP = vdet.PROTOCOL["step"]          # train frame step, from Section 8 and the GPU budget
VAL_STEP = vdet.PROTOCOL["val_step"]  # same validation subsample for every model
yaml_path = vdet.build_yolo_dataset(imgs, anns, folds, val_fold=0,
                                    out_dir=OUT / "yolo_fold0", step=STEP, val_step=VAL_STEP, mask_ignore=True)
print(yaml_path.read_text())

In [ ]:
# Visual check of the masking: original vs exported training image
masked = list((OUT / "yolo_fold0" / "images" / "train").rglob("*.jpg"))
masked = [p for p in masked if not p.is_symlink()]
if masked:
    p = masked[0]; clip_uid = p.parent.name.replace("__", "/")
    row = next(imgs[(imgs.clip_uid == clip_uid) & (imgs.frame == p.stem)].itertuples())
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    show_frame(row, axes[0])
    axes[1].imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)); axes[1].set_title("exported (masked)"); axes[1].axis("off")
    plt.tight_layout(); plt.show()

**Resizing / normalisation / augmentation (applied inside each detector's framework, fixed seed):** letterbox resize of the long side to `IMGSZ`, ImageNet mean/std normalisation for torchvision/HF models (Ultralytics uses /255), horizontal flip, mild HSV jitter and scale jitter. No vertical flip or rotation by default: drone footage has a consistent "up" for vehicles and people only partly, so these are tested as experiments rather than assumed. Mosaic is evaluated as an experiment because it shrinks already tiny objects further.

## 12. Evaluator sanity check

In [ ]:
# Ground truth as predictions -> AP must be 1.0; jittering boxes shows how harsh IoU is for tiny objects.
val_imgs = vdet.subsample(fi[fi.fold == 0], VAL_STEP)
vg = gt[gt.key.isin(val_imgs.key)].assign(score=1.0)
print("GT as predictions:", {k: round(v, 3) for k, v in vdet.coco_evaluate(val_imgs, anns, vg, per_class=False).items()})
for px in [1, 2, 4]:
    j = vg.copy(); j[["x", "y"]] += np.random.default_rng(0).uniform(-px, px, (len(j), 2))
    r = vdet.coco_evaluate(val_imgs, anns, j, per_class=False)
    print(f"shift ±{px}px: AP={r['AP']:.3f} AP50={r['AP50']:.3f} AP_small={r['AP_small']:.3f}")
# detections placed inside ignored regions must not change the score
iv = ign[ign.key.isin(val_imgs.key)]
fake = pd.concat([iv.assign(cls=c, score=0.99) for c in range(len(CLASSES))])
r = vdet.coco_evaluate(val_imgs, anns, pd.concat([vg, fake]), per_class=False)
print(f"GT + {len(fake)} detections inside ignored regions: AP={r['AP']:.3f}")

In [ ]:
# Empty submission round-trip (format check of the I/O helpers)
empty = pd.DataFrame(columns=["image_id", "cls", "score", "x", "y", "w", "h"])
vdet.write_submission(empty, ROOT / "test_images.csv", OUT / "submission_empty.csv").head(3)

## 13. Summary of observations → decisions

In [ ]:
summary = {
    "frames": len(imgs), "boxes": len(gt), "clips": imgs.clip_uid.nunique(),
    "class_share": gt.cls_name.value_counts(normalize=True).round(4).to_dict(),
    "small_share": float(overall["small"]), "median_sqrt_area": float(gt.sqrt_area.median()),
    "obj_per_frame_median": float(imgs.n_obj.median()), "obj_per_frame_p99": float(imgs.n_obj.quantile(.99)),
    "frames_with_ignore": float(len(ign_frames) / len(imgs)), "anchors_mean_iou": mean_iou,
    "step": STEP, "imgsz": IMGSZ,
}
json.dump(summary, open(OUT / "eda_summary.json", "w"), indent=1); summary

Fill in with the numbers above before writing the report:

| Observation | Decision |
|---|---|
| ~X% of boxes are COCO-small; median side Y px | high input resolution (`IMGSZ`), P2 head and SAHI as experiments; report AP_small everywhere |
| strong class imbalance (car/person dominate, tricycle/bus rare) | track per-class AP; try focal-loss γ / copy-paste for rare classes |
| up to Z objects per frame, some > 300 | `max_det=500` at inference; DETR queries ≥ 300 (tune vs density) |
| boxes overlapping at IoU>0.5: W% | NMS IoU threshold and Soft-NMS as experiments |
| consecutive frames near-duplicates | train on every `STEP`-th frame; temporal smoothing bonus |
| ignored regions in V% of frames | mask in training, crowd in evaluation |
| resolution mix differs train vs test (if it does) | check per-resolution validation AP |